# Hardware Mapping Tutorial

Partition, allocate, route, and place TALON IR graphs on multi-core hardware.

In [ ]:
import numpy as np
from talon import ir, graph as talongraph

## 1. HardwareSpec

In [ ]:
spec = talongraph.HardwareSpec(
    max_neurons_per_core=1024, max_synapses_per_core=200000,
    sram_bytes_per_core=524288, num_cores=4, max_feedback_delay=3,
)
print(f"Cores: {spec.num_cores}, SRAM: {spec.sram_bytes_per_core:,}")
print(f"Presets: Zynq-7020={talongraph.HardwareSpec.zynq_7020().num_cores} cores, ZCU102={talongraph.HardwareSpec.zcu102().num_cores} cores")

## 2. Graph

In [ ]:
nodes = {
    "input": ir.Input(np.array([784])),
    "fc1": ir.Affine(weight=np.random.randn(64, 784).astype(np.float32)*0.01, bias=np.zeros(64, dtype=np.float32)),
    "lif1": ir.LIF(tau=np.ones(64, dtype=np.float32)*10, r=np.ones(64, dtype=np.float32), v_leak=np.zeros(64, dtype=np.float32), v_threshold=np.ones(64, dtype=np.float32)),
    "fc2": ir.Affine(weight=np.random.randn(10, 64).astype(np.float32)*0.01, bias=np.zeros(10, dtype=np.float32)),
    "lif2": ir.LIF(tau=np.ones(10, dtype=np.float32)*10, r=np.ones(10, dtype=np.float32), v_leak=np.zeros(10, dtype=np.float32), v_threshold=np.ones(10, dtype=np.float32)),
    "output": ir.Output(np.array([10])),
}
edges = [("input","fc1"),("fc1","lif1"),("lif1","fc2"),("fc2","lif2"),("lif2","output")]
graph = ir.Graph(nodes=nodes, edges=edges)
print(f"Graph: {len(graph.nodes)} nodes")

## 3. Partition -> Allocate -> Route -> Place

In [ ]:
p = talongraph.partition(graph, spec)
pm = p.partition_metadata
print(f"Partition: {pm['num_cores_used']} cores")
for n, c in pm["assignments"].items():
    print(f"  {n} -> core {c}")

In [ ]:
res = talongraph.allocate(p, spec)
print(f"Fits: {res.fits_hardware}, peak: {res.peak_core_utilization:.1%}")

In [ ]:
routed = talongraph.route(p, spec)
print(f"Routed: {len(routed.edges)} edges")

In [ ]:
pl = talongraph.place(p, spec)
print(f"Hops: {pl.total_hop_distance}, improvement: {pl.improvement:.1f}%")
for c, pos in pl.logical_to_physical.items():
    print(f"  Core {c} -> ({pos[0]}, {pos[1]})")